# Combine Chromosomes

In [1]:
%%bash

#!/bin/bash

set -o pipefail
set -o errexit

# Grab existing run info
source "/home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_results/latest_run.env"

echo "Combining outputs from ${OUTPUT_PATH}"

log_file="${OUTPUT_PATH}/combine_chr_${PGS}_${RUN_ID}_log.txt"
exec > >(tee -a "${log_file}") 2>&1

combined_file="${OUTPUT_PATH}/${PGS}_combined_chr1_22_${RUN_ID}.csv"

sscore_files=()

for chromo in {1..22}; do
    file_name="${PGS}_score_file_chr${chromo}.sscore"
    local_path="${OUTPUT_PATH}/${file_name}"

    if [[ -f "${local_path}" ]]; then
        sscore_files+=("${local_path}")
    else
        echo "Missing ${local_path}"
    fi
done

if [[ ${#sscore_files[@]} -eq 0 ]]; then
    echo "ERROR: No .sscore files found in ${OUTPUT_PATH}"
    exit 1
fi

awk '
    BEGIN {
        OFS = ","
        print "IID,CNT,CNT2,SCORE"
    }

    FNR == 1 {
        next
    }

    {
        iid = $2
        cnt[iid] += $3
        cnt2[iid] += $4
        score[iid] += $5
    }

    END {
        for (iid in score) {
            print iid, cnt[iid], cnt2[iid], score[iid]
        }
    }
' "${sscore_files[@]}" > "${combined_file}"

echo "Combined file written to ${combined_file}"

Combining outputs from /home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_results/PGS002308_GRCh38_20260909_135450
Combined file written to /home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_results/PGS002308_GRCh38_20260909_135450/PGS002308_combined_chr1_22_20260909_135450.csv


# Check QC logs

In [2]:
%%bash

#!/bin/bash

set -o pipefail
set -o errexit

source "/home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_results/latest_run.env"

BASE_DIRECTORY="/home/jupyter/workspace/workspace-bucket/calculate_pgs"
WEIGHTS_PATH="${BASE_DIRECTORY}/pgs_bim_matched_weights"

echo "Reading PLINK logs from ${OUTPUT_PATH}"

summary_csv="${OUTPUT_PATH}/variants_incorporated_by_chr_${PGS}_${RUN_ID}.csv"
echo "chromosome,valid_predictors_loaded,log_file" > "${summary_csv}"

shopt -s nullglob

score_files=( "${WEIGHTS_PATH}/${PGS}"*plink_score*"${BUILD}"_prepared.txt )

if (( ${#score_files[@]} == 0 )); then
    echo "ERROR: No prepared score file found in ${WEIGHTS_PATH}"
    echo "Looked for: ${WEIGHTS_PATH}/${PGS}*plink_score*${BUILD}_prepared.txt"
    exit 1
fi

score_file="${score_files[0]}"
total_possible="$(wc -l < "${score_file}")"

echo "Prepared score file: ${score_file}"
echo "Total possible matched variants: ${total_possible}"

declare -A processed_chromosomes
total_predictors=0

for log_file in "${OUTPUT_PATH}"/plink*.log; do
    base_name="$(basename "${log_file}")"

    if [[ "${base_name}" =~ chr([0-9]+|X|Y)\.log$ ]]; then
        chromosome="chr${BASH_REMATCH[1]}"
    else
        echo "Chromosome not found in filename: ${base_name}"
        continue
    fi

    if [[ -n "${processed_chromosomes[$chromosome]:-}" ]]; then
        echo "Already processed ${chromosome}; skipping ${base_name}"
        continue
    fi

    processed_chromosomes["${chromosome}"]=1

    predictor_count="$(
        grep -oE '[0-9]+ variants processed' "${log_file}" \
            | tail -n 1 \
            | grep -oE '^[0-9]+' \
            || echo 0
    )"

    total_predictors=$((total_predictors + predictor_count))

    echo "${chromosome}: ${predictor_count} variants processed"
    echo "${chromosome},${predictor_count},${base_name}" >> "${summary_csv}"
done

echo "TOTAL valid predictors loaded: ${total_predictors}"
echo "TOTAL,${total_predictors},all_logs" >> "${summary_csv}"
echo "TOTAL_POSSIBLE_MATCHED_VARIANTS,${total_possible},${score_file##*/}" >> "${summary_csv}"

percent_incorporated="$(
    awk -v used="${total_predictors}" -v possible="${total_possible}" \
        'BEGIN {
            if (possible > 0) {
                printf "%.2f", 100 * used / possible
            } else {
                print "NA"
            }
        }'
)"

echo "PERCENT_INCORPORATED,${percent_incorporated},percent_of_possible_matched_variants" >> "${summary_csv}"

echo "Wrote summary to ${summary_csv}"

Reading PLINK logs from /home/jupyter/workspace/workspace-bucket/calculate_pgs/plink_results/PGS002308_GRCh38_20260909_135450
Prepared score file: /home/jupyter/workspace/workspace-bucket/calculate_pgs/pgs_bim_matched_weights/PGS002308_plink_score_GRCh38_prepared.txt
Total possible matched variants: 1258286
chr10: 67213 variants processed
chr11: 64289 variants processed
chr12: 62589 variants processed
chr13: 47688 variants processed
chr14: 41665 variants processed
chr15: 37965 variants processed
chr16: 38905 variants processed
chr17: 34267 variants processed
chr18: 37786 variants processed
chr19: 23112 variants processed
chr1: 105098 variants processed
chr20: 32977 variants processed
chr21: 17456 variants processed
chr22: 18273 variants processed
chr2: 106017 variants processed
chr3: 87755 variants processed
chr4: 78699 variants processed
chr5: 78676 variants processed
chr6: 83511 variants processed
chr7: 68460 variants processed
chr8: 68125 variants processed
chr9: 57760 variants proc